In [1]:
using PVlib
using Dates
using Plots
using StatsPlots
using CSV

In [ ]:
# latitude, longitude, name, altitude, timezone
coordinates = [
    (lat=32.2, lon=-111.0, name="Tucson", elevation=700, tz=TimeZone("America/Denver")),
    (lat=35.1, lon=-106.6, name="Albuquerque", elevation=1500, tz=TimeZone("America/Denver")),
    (lat=37.8, lon=-122.4, name="San Francisco", elevation=10, tz=TimeZone("America/Chicago")),
    (lat=52.5, lon=13.4, name="Berlin", elevation=34, tz=TimeZone("Europe/Berlin")),
]

In [ ]:
module_filename = "sam-library-sandia-modules-2015-6-30.csv"
module_name = "Canadian Solar CS5P-220M [ 2009]"  # "BP Solar SX305 [2007 (E)]"

inverter_filename = "sam-library-cec-inverters-2019-03-05.csv"
inverter_name = "ABB: MICRO-0.25-I-OUTD-US-208 [208V]" # "SMA America: SB 240-US-10 [240V]"

pv_module = read_solar_module(module_name, module_filename)
pv_inverter = read_solar_inverter(inverter_name, inverter_filename)

In [ ]:
for (latitude, longitude, name, elevation, tz) in coordinates
    println("City: $name, lat=$latitude, lon=$longitude, elevation=$elevation, tz=$tz")
end

In [ ]:
ac_power_totals = Float64[]

for (latitude, longitude, name, elevation, tz) in coordinates

    surface_tilt = latitude
    surface_azimuth = 180
    albedo = 0.25

    weather_data = get_meteorological_data_pvgis(latitude, longitude, Date(1900, 1, 1), Date(2026, 1, 1), tz, false)
    solar_position = get_solar_position(latitude, longitude, elevation, weather_data)
    total_irradiance = get_total_irradiance(surface_tilt, surface_azimuth, weather_data, solar_position, albedo) # verified same
    cell_temp = sapm_cell_temperature(total_irradiance, weather_data) # verified same
    effective_irradiance = sapm_effective_irradiance(total_irradiance, pv_module, solar_position, surface_tilt, surface_azimuth, elevation) # verified same
    dc_components = sapm_dc_components(pv_module, effective_irradiance, cell_temp)
    ac_power = sandia_ac_power(pv_inverter, dc_components) # verified same

    ac_power_total = sum(getfield.(ac_power,:ac_power))
    push!(ac_power_totals, ac_power_total)
end

In [ ]:
py_results = CSV.File("../data/python_validation_result.csv")
py_energy = py_results.yearly_energy

names = getfield.(coordinates, :name)

p = groupedbar(
    names,
    [ac_power_totals py_energy],
    ylabel = "yearly energy yield (Wh)",
    label = ["PVlib.jl" "pvlib-python"],
    yformatter = y -> string(round(Int, y)),
)
display(p)

#savefig(p, "pvlib_validation.png")

In [ ]:
percent_error = 100 .* abs.(ac_power_totals .- py_energy) ./ abs.(py_energy)

println("Energy yield and percentage error across locations:")
for (name, ac_total, py_total, err) in zip(names, ac_power_totals, py_energy, percent_error)
    println("$name:")
    println("  PVlib.jl AC power total: $(round(ac_total, digits=2))")
    println("  pvlib-python energy:     $(round(py_total, digits=2))")
    println("  Percentage error:        $(round(err, digits=2)) %")
end